<a href="https://colab.research.google.com/github/navyaarora235/AI-Internship-Solutions/blob/main/AI_Internship_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-generativeai

In [2]:
import os
import google.generativeai as genai

# Quick configuration helper. Paste your actual API key here.
# For testing, you can grab a free API key from Google AI Studio.
API_KEY = "YOUR_GEMINI_API_KEY_HERE"
genai.configure(api_key=API_KEY)

# SYSTEM PROMPTS & KNOWLEDGE BASE
# Giving the AI distinct context about the NGO so it doesn't hallucinate
NGO_CONTEXT = """
You are a brilliant support assistant for 'EcoReach', an NGO focused on clean water initiatives.
- If the user is a DONOR: Explain that 92% of all donations go straight to field water filters. Projects are active in Kenya and India. A $25 donation provides clean water to a family for one year.
- If the user is a BENEFICIARY (seeking help): Tell them water distributions happen every Tuesday at 9 AM at the regional community center. Applications can be submitted online.
- Maintain an incredibly empathetic, professional, and clear tone. Always answer in the exact language the user used to ask the question.
"""

# Simple dictionary to serve as our "In-Memory Session Storage"
# This tracks conversation turns so the chatbot has a functional memory.
conversation_history = []

# MULTI-STEP WORKFLOW ENGINE

def analyze_user_intent(user_message):
    """
    Step 1 in the workflow: Classifies what the user wants before generating a response.
    This ensures the NGO routes resources correctly.
    """
    classification_model = genai.GenerativeModel('gemini-1.5-flash')

    prompt = f"""
    Analyze this message from an NGO contact: "{user_message}"
    Classify it strictly into one of these categories: [DONOR_INQUIRY, BENEFICIARY_HELP, GENERAL_GREETING].
    Return only the category name as a single word.
    """
    try:
        response = classification_model.generate_content(prompt)
        return response.text.strip()
    except Exception:
        return "GENERAL_GREETING"


def generate_ngo_response(user_message, intent, history):
    """
    Step 2 & 3 in the workflow: Compiles the history context and builds the final tailored response.
    """
    response_model = genai.GenerativeModel('gemini-1.5-flash')

    # Formatting our memory array into a clean block of text the model can read
    formatted_memory = ""
    for turn in history[-4:]: # Only grab the last 4 turns to prevent token bloating
        formatted_memory += f"User: {turn['user']}\nAI: {turn['ai']}\n"

    final_prompt = f"""
    {NGO_CONTEXT}

    Here is the conversation history so far:
    {formatted_memory}

    The backend classified the user's current intent as: {intent}
    Current User Message: "{user_message}"

    Draft the final helpful response now:
    """

    response = response_model.generate_content(final_prompt)
    return response.text.strip()

# CONVERSATION LOOP TERMINAL (Interactive Test Environment)
def run_support_agent():
    print("=== EcoReach NGO AI Support System Initialized ===")
    print("Type 'exit' to end the session.\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Closing support session. Context wiped.")
            break

        if not user_input.strip():
            continue

        # Step 1: Run classification workflow
        detected_intent = analyze_user_intent(user_input)

        # Step 2 & 3: Pull memory and generate response
        ai_response = generate_ngo_response(user_input, detected_intent, conversation_history)

        # Step 4: Update the memory feature storage
        conversation_history.append({
            "user": user_input,
            "ai": ai_response
        })

        print(f"\n[System Intent Log: {detected_intent}]")
        print(f"EcoReach Agent: {ai_response}\n")

# To test this right inside your notebook, uncomment the line below and run the cell:
run_support_agent()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


=== EcoReach NGO AI Support System Initialized ===
Type 'exit' to end the session.

You: exit
Closing support session. Context wiped.
